# 02 – Preprocessing
**Owner:** Tony  
**Goal:** Clean the data, engineer home-minus-away features, and export the processed dataset used for modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
FIGURES = Path("../reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
games = pd.read_csv(RAW / "games.csv", parse_dates=["GAME_DATE_EST"])
print(games.shape)

## Remove duplicates and missing rows

In [ ]:
games = games.drop_duplicates(subset="GAME_ID")
games = games.dropna(subset=[c for c in games.columns if c.endswith(("_home", "_away"))])
print(games.shape)

## Regular season filter
NBA `GAME_ID`s starting with `2` are regular-season games. Verify this against the data before relying on it.

In [ ]:
games = games[games["GAME_ID"].astype(str).str.startswith("2")]
print(games.shape)

## Feature engineering: home minus away differences

In [ ]:
stats = ["FG_PCT", "FT_PCT", "FG3_PCT", "AST", "REB"]
for s in stats:
    games[f"{s}_diff"] = games[f"{s}_home"] - games[f"{s}_away"]

# Points are excluded as model features because they directly determine the winner (target leakage).
feature_cols = [f"{s}_diff" for s in stats]
processed = games[["GAME_ID", "GAME_DATE_EST", "SEASON"] + feature_cols + ["HOME_TEAM_WINS"]]
processed.head()

## Export

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)
processed.to_csv(PROCESSED / "games_processed.csv", index=False)
print("Saved", processed.shape)